In [1]:
!pip install pulp

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
   ------- -------------------------------- 3.1/16.4 MB 11.9 MB/s eta 0:00:02
   ----------------- ---------------------- 7.3/16.4 MB 16.0 MB/s eta 0:00:01
   ---------------------------- ----------- 11.8/16.4 MB 18.0 MB/s eta 0:00:01
   -------------------------------------- - 15.7/16.4 MB 18.6 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 17.4 MB/s  0:00:01


In [5]:
import sqlite3
import pandas as pd
import pulp

ALDI_DB = "aldi_products.db"

def load_linked_grocery_data():
    conn = sqlite3.connect(ALDI_DB)
    
    # 1. Enhanced Query: Join categories so we can filter non-human food groups
    query = """
    SELECT 
        p.sku,
        p.name AS product_name,
        p.price_pennies,
        p.selling_size,
        n.food_name,
        n.energy_kcal,
        n.protein_g,
        n.fat_g,
        n.carbohydrate_g,
        m.match_confidence,
        GROUP_CONCAT(c.category_name, ' | ') as all_categories
    FROM product_nutrition_matches m
    JOIN products p ON m.sku = p.sku
    JOIN nutrients n ON m.food_code = n.food_code
    LEFT JOIN product_categories c ON p.sku = c.sku
    WHERE p.price_pennies > 0 
      AND p.selling_size IS NOT NULL
      AND m.match_confidence >= 75.0  -- CRITICAL: Raised floor from 50 to 75 to drop bad links
    GROUP BY p.sku
    """
    
    df = pd.read_sql(query, conn)
    conn.close()
    
    optimized_items = {}
    import re
    weight_pattern = re.compile(r"([\d\.]+)\s*(kg|g|l|ml|pack)", re.IGNORECASE)
    
    # Define keywords that immediately flag an item as non-human food
    blocklist_keywords = ['dog', 'cat', 'pet', 'wipes', 'bleach', 'diaper', 'nappy', 'shampoo', 'nappies']
    
    for idx, row in df.iterrows():
        product_name = str(row['product_name']).lower()
        categories_str = str(row['all_categories']).lower()
        
        # Guardrail A: Strip out non-food items caught by text mismatches
        if any(kw in product_name for kw in blocklist_keywords) or any(kw in categories_str for kw in blocklist_keywords):
            continue
            
        size_str = str(row['selling_size'])
        match = weight_pattern.search(size_str)
        if not match: continue
            
        value = float(match.group(1))
        unit = match.group(2).lower()
        
        if unit in ['kg', 'l']:
            total_weight_grams = value * 1000
        else:
            total_weight_grams = value
            
        if total_weight_grams <= 0: continue
            
        cost_pounds = row['price_pennies'] / 100.0
        cost_per_100g = (cost_pounds / total_weight_grams) * 100
        
        # Guardrail B: If cost calculation returns an absurd math error, drop it
        if cost_per_100g > 50.0: # If it costs more than £50 per 100g, it's likely an error or a non-food item
            continue
            
        clean_id = f"SKU_{row['sku']}_" + re.sub(r'[^a-zA-Z0-9]', '_', row['product_name'])[:40]
        
        optimized_items[clean_id] = {
            "sku": row['sku'],
            "display_name": row['product_name'],
            "cost_per_100g": cost_per_100g,
            "calories": float(row['energy_kcal']),
            "protein": float(row['protein_g']),
            "fat": float(row['fat_g']),
            "carbs": float(row['carbohydrate_g'])
        }
        
    return optimized_items

def optimize_diet(target_calories, min_protein, max_fat, min_carbs):
    """
    Builds and solves the Linear Programming matrix using PuLP.
    """
    # 1. Fetch available item maps
    food_pool = load_linked_grocery_data()
    
    if not food_pool:
        print("No viable connected product profiles found in database. Check your table matches.")
        return
        
    # 2. Define the Minimization Problem
    prob = pulp.LpProblem("Cheapest_Diet_Optimization", pulp.LpMinimize)
    
    # 3. Decision Variables: represents amount in 100g units to eat of that specific item per day
    # We apply an upper bound of 3.0 (300g max per single food item) to ensure diversity 
    # and prevent the model from telling you to eat 2.5kg of sugar or pure flour.
    food_vars = pulp.LpVariable.dicts(
        "100g_Units", 
        food_pool.keys(), 
        lowBound=0, 
        upBound=3.0, 
        cat='Continuous'
    )
    
    # 4. Objective Function: Minimize overall monetary expenditure
    prob += pulp.lpSum([food_pool[item]["cost_per_100g"] * food_vars[item] for item in food_pool])
    
    # 5. Adding Daily Target Nutrient Boundary Constraints
    # Energy Constraint
    prob += pulp.lpSum([food_pool[item]["calories"] * food_vars[item] for item in food_pool]) >= target_calories
    
    # Protein Floor
    prob += pulp.lpSum([food_pool[item]["protein"] * food_vars[item] for item in food_pool]) >= min_protein
    
    # Carbohydrates Floor
    prob += pulp.lpSum([food_pool[item]["carbs"] * food_vars[item] for item in food_pool]) >= min_carbs
    
    # Fat Ceiling Boundary
    prob += pulp.lpSum([food_pool[item]["fat"] * food_vars[item] for item in food_pool]) <= max_fat

    print("\nCalculating optimal diet combinations via matrix solver...")
    
    # 6. Execute the LP Optimization Solver
    # Using msg=False keeps the backend matrix algebra printout hidden
    status = prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # 7. Format Output Metrics Summary
    print(f"Solver Outcome Status: {pulp.LpStatus[status]}")
    
    if pulp.LpStatus[status] != "Optimal":
        print("❌ Could not find a feasible food balance meeting all constraints simultaneously. Try loosening target values.")
        return

    print("\n================ THE CHEAPEST DAILY DIET GRID ================")
    total_cost = 0.0
    
    # Initialize dictionary to dynamically accumulate every single metric we track
    diet_totals = {
        "calories": 0.0,
        "protein": 0.0,
        "fat": 0.0,
        "carbs": 0.0,
        "fibre": 0.0,   # Micro / CoFID proximate addition
        "sodium": 0.0   # Micro / CoFID proximate addition
    }
    
    for item_id, variable in food_vars.items():
        units_to_consume = variable.varValue
        if units_to_consume and units_to_consume > 0.01:
            meta = food_pool[item_id]
            weight_grams = units_to_consume * 100
            item_cost = meta["cost_per_100g"] * units_to_consume
            total_cost += item_cost
            
            # Dynamically add up all macros and micros based on the 100g multiplier
            for nutrient in diet_totals.keys():
                # Safely grab the value from our pool (defaulting to 0 if not found)
                diet_totals[nutrient] += meta.get(nutrient, 0.0) * units_to_consume
            
            print(f" -> Eat: {weight_grams:6.1f}g | {meta['display_name'][:50]:<50} (Cost: £{item_cost:.2f})")
            
    print("==============================================================")
    print(f" DAILY DIET BUDGET COST : £{total_cost:.2f}")
    
    # --- New Detailed Nutrition Summary Block ---
    print("\n================ FINAL DIET NUTRITION SUMMARY ================")
    print(f" Core Macronutrients:")
    print(f"   • Energy Yield : {diet_totals['calories']:7.1f} kcal / target {target_calories} kcal")
    print(f"   • Protein      : {diet_totals['protein']:7.1f}g    / target min {min_protein}g")
    print(f"   • Carbohydrates: {diet_totals['carbs']:7.1f}g    / target min {min_carbs}g")
    print(f"   • Total Fat    : {diet_totals['fat']:7.1f}g    / target max {max_fat}g")
    print(f"--------------------------------------------------------------")
    print(f" Micronutrients & Dietary Markers (Per Day):")
    print(f"   • Dietary Fibre: {diet_totals['fibre']:7.1f}g")
    print(f"   • Sodium       : {diet_totals['sodium']:7.1f}mg")
    print("==============================================================")

# --- Run Optimization Execution Engine ---
if __name__ == "__main__":
    # Define targets: Calories (kcal), Protein (g), Max Fat (g), Min Carbs (g)
    TARGET_CALORIES = 2200
    MIN_PROTEIN = 150
    MAX_FAT = 80
    MIN_CARBS = 200
    
    optimize_diet(TARGET_CALORIES, MIN_PROTEIN, MAX_FAT, MIN_CARBS)


Calculating optimal diet combinations via matrix solver...
Solver Outcome Status: Optimal

================ THE CHEAPEST DAILY DIET GRID ================
 -> Eat:  286.3g | Chicken Drumsticks                                 (Cost: £0.62)
 -> Eat:   83.1g | Rich Tea Biscuits                                  (Cost: £0.14)
 -> Eat:  228.8g | Digestive Biscuits                                 (Cost: £0.34)
 -> Eat:  300.0g | Sliced Mushrooms                                   (Cost: £0.59)
 DAILY DIET BUDGET COST : £1.68

================ FINAL DIET NUTRITION SUMMARY ================
 Core Macronutrients:
   • Energy Yield :  2200.0 kcal / target 2200 kcal
   • Protein      :   150.0g    / target min 150g
   • Carbohydrates:   234.3g    / target min 200g
   • Total Fat    :    80.0g    / target max 80g
--------------------------------------------------------------
 Micronutrients & Dietary Markers (Per Day):
   • Dietary Fibre:     0.0g
   • Sodium       :     0.0mg
